In [1]:
import requests
import pandas as pd
import time

all_subfields = []
page = 1

while True:
    url = f"https://api.openalex.org/subfields?per_page=200&page={page}"
    response = requests.get(url)
    
    if response.status_code != 200:
        break
        
    data = response.json().get('results', [])
    if not data:
        break
        
    all_subfields.extend(data)
    print(f"  - Fetched page {page} ({len(data)} subfields)")
    
    page += 1
    time.sleep(0.5) 

hierarchy_map = {}
for sf in all_subfields:
    sf_id = sf['id'].split('/')[-1] # e.g., "1100"
    
    field = sf.get('field', {})
    domain = sf.get('domain', {})
    
    hierarchy_map[sf_id] = {
        'Subfield': sf['display_name'],
        'ID': field.get('id', '').split('/')[-1],
        'Field': field.get('display_name', ''),
        'Domain': domain.get('display_name', '')
    }

print(f"Successfully built hierarchy for {len(hierarchy_map)} subfields.")
pd.DataFrame.from_dict(hierarchy_map, orient='index').head()


  - Fetched page 1 (200 subfields)
  - Fetched page 2 (52 subfields)
Successfully built hierarchy for 252 subfields.


,Subfield,ID,Field,Domain
3312,Sociology and Political Science,33,Social Sciences,Social Sciences
3106,Nuclear and High Energy Physics,31,Physics and Astronomy,Physical Sciences
1110,Plant Science,11,Agricultural and Biological Sciences,Life Sciences
1312,Molecular Biology,13,"Biochemistry, Genetics and Molecular Biology",Life Sciences
2208,Electrical and Electronic Engineering,22,Engineering,Physical Sciences


In [2]:
import collections

uva_id = "I51556381"
select_fields = "id,primary_topic"
filter_str = f"authorships.institutions.lineage:{uva_id},publication_year:2025"

cursor = "*"
subfield_counts = collections.Counter()
total_works = 0

while True:
    url = (
        f"https://api.openalex.org/works?"
        f"filter={filter_str}"
        f"&select={select_fields}"
        f"&per_page=200&cursor={cursor}"
    )
    
    try:
        r = requests.get(url)
        r.raise_for_status() # Raise error for 400/500 codes
        data = r.json()
        
        results = data.get('results', [])
        if not results:
            break
            
        # Update cursor for next loop
        cursor = data['meta']['next_cursor']
        
        # Count the subfields in this batch
        for work in results:
            topic = work.get('primary_topic')
            if topic and topic.get('subfield'):
                sf_id = topic['subfield']['id'].split('/')[-1]
                subfield_counts[sf_id] += 1
        
        total_works += len(results)
        print(f"  - Processed {total_works} works...")
        
    except Exception as e:
        print(f"Error fetching data: {e}")
        break

print(f"Finished. Analyzed {total_works} papers across {len(subfield_counts)} subfields.")

  - Processed 200 works...
  - Processed 400 works...
  - Processed 600 works...
  - Processed 800 works...
  - Processed 1000 works...
  - Processed 1200 works...
  - Processed 1400 works...
  - Processed 1600 works...
  - Processed 1800 works...
  - Processed 2000 works...
  - Processed 2200 works...
  - Processed 2400 works...
  - Processed 2600 works...
  - Processed 2800 works...
  - Processed 3000 works...
  - Processed 3200 works...
  - Processed 3400 works...
  - Processed 3600 works...
  - Processed 3800 works...
  - Processed 4000 works...
  - Processed 4200 works...
  - Processed 4400 works...
  - Processed 4600 works...
  - Processed 4800 works...
  - Processed 5000 works...
  - Processed 5200 works...
  - Processed 5400 works...
  - Processed 5600 works...
  - Processed 5800 works...
  - Processed 6000 works...
  - Processed 6200 works...
  - Processed 6400 works...
  - Processed 6600 works...
  - Processed 6800 works...
  - Processed 7000 works...
  - Processed 7200 works

In [3]:

rows = []
for sf_id, count in subfield_counts.items():
    info = hierarchy_map.get(sf_id)
    if info:
        rows.append({
            'Domain': info['Domain'],
            'Group': info['Field'],
            'Field': info['Subfield'],
            'numpub': count,
        })

df = pd.DataFrame(rows)

# Sort Hierarchy: Domain -> Group -> Field
df = df.sort_values(by=['Domain', 'Group', 'Field'])

print(f"\nPreview of Results ({len(df)} rows):")
df.head()


Preview of Results (228 rows):


,Domain,Group,Field,numpub
212,Health Sciences,Dentistry,Oral Surgery,1
122,Health Sciences,Dentistry,Periodontics,1
223,Health Sciences,Health Professions,Complementary and Manual Therapy,1
91,Health Sciences,Health Professions,Emergency Medical Services,12
38,Health Sciences,Health Professions,General Health Professions,79


In [4]:


df = df.sort_values(by=['Domain', 'Group', 'Field'])
df = df[['Domain', 'Group', 'Field', 'numpub']]

print(f"\nPreview of Results ({len(df)} rows):")
df.head()


Preview of Results (228 rows):


,Domain,Group,Field,numpub
212,Health Sciences,Dentistry,Oral Surgery,1
122,Health Sciences,Dentistry,Periodontics,1
223,Health Sciences,Health Professions,Complementary and Manual Therapy,1
91,Health Sciences,Health Professions,Emergency Medical Services,12
38,Health Sciences,Health Professions,General Health Professions,79


In [5]:
# Export to CSV
df.to_csv("uva_2025_full_hierarchy.csv", index=False)